# **🚀 FLEX-Med: Local Training Orchestrator**

This notebook provides a robust API-driven orchestrator for training local models. It is designed to be triggered by the FLEX-Med backend and can update its status directly in the Supabase database.

## **1. Setup & Configuration**

In [ ]:
from google.colab import drive
import os
import sys
import json
import datetime
import logging
import subprocess
import threading
import traceback
import io
import base64
import time
from typing import List, Tuple, Dict, Optional
from pathlib import Path

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torchvision.models import get_model, list_models
from torch.utils.data import DataLoader

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException, BackgroundTasks, File, UploadFile
from fastapi.responses import FileResponse, PlainTextResponse, JSONResponse
from pydantic import BaseModel
from starlette.middleware.cors import CORSMiddleware
from pyngrok import ngrok
from pyngrok.process import kill_process
from pyngrok.exception import PyngrokNgrokError

# Supabase
try:
    from supabase import create_client, Client
except ImportError:
    print("⚠ Supabase library not found. Installing in next cell...")

nest_asyncio.apply()

# --- DRIVE & PATH SETUP ---
if os.path.exists('/content/drive'):
    print("✓ Drive already mounted")
else:
    drive.mount('/content/drive')

POSSIBLE_ROOTS = [
    "/content/drive/MyDrive/College/FLEX-Med/flex-med",
    "/content/drive/MyDrive/FLEX-Med/flex-med"
]

PROJECT_ROOT = None
for root in POSSIBLE_ROOTS:
    if os.path.exists(root):
        PROJECT_ROOT = root
        break

if PROJECT_ROOT:
    %cd {PROJECT_ROOT}
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    print(f"✓ Switched to project root: {PROJECT_ROOT}")
else:
    print("⚠ WARNING: Project root not found in known locations.")
    PROJECT_ROOT = os.getcwd()

# Import centralized config
try:
    from flex_med.utils.config import MODELS_PATH, CLIENT_INFO_FILE_PATH, PROJECT_DIR
    MODELS_DIR = MODELS_PATH
    print("✓ Centralized configuration loaded")
except ImportError:
    print("⚠ flex_med module not found. Path issues?")
    MODELS_DIR = Path("/content/drive/MyDrive/College/models")
    PROJECT_DIR = Path(PROJECT_ROOT)

os.makedirs(MODELS_DIR, exist_ok=True)

In [ ]:
# Dependency Installation
!pip install fastapi uvicorn pyngrok nest-asyncio supabase torch torchvision grad-cam lime scikit-image pydantic-settings -q
print("✅ Dependencies installed")

## **2. Ngrok Cleanup**

In [ ]:
print("🧹 Cleaning up existing ngrok sessions...")
try:
    ngrok.kill()
    time.sleep(1)
except:
    pass

try:
    subprocess.run(["fuser", "-k", "8000/tcp"], check=False, capture_output=True)
    time.sleep(1)
except:
    pass
print("✅ Cleanup complete!")

## **3. API & Orchestrator Implementation**

In [ ]:
# ==========================================
# 1. LOGGING SYSTEM
# ==========================================
CENTRAL_LOG_FILE = "local_training.log"
training_logs = {}

def log_message(client_name: str, message: str, level: str = "INFO"):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    formatted_msg = f"[{timestamp}] [{level}] [{client_name}] {message}"
    
    # 1. Console
    print(formatted_msg)
    
    # 2. Central Log File
    with open(CENTRAL_LOG_FILE, "a") as f:
        f.write(formatted_msg + "\n")
        f.flush()
        
    # 3. Per-client log file
    client_log_file = f"{client_name}.log"
    with open(client_log_file, "a") as f:
        f.write(f"[{timestamp}] {message}\n")
        f.flush()
        
    # 4. In-memory logs
    if client_name not in training_logs:
        training_logs[client_name] = {"status": "starting", "logs": []}
    training_logs[client_name]["logs"].append(f"[{timestamp}] {message}")

# Initialize central log
with open(CENTRAL_LOG_FILE, "w") as f:
    f.write(f"===== LOCAL TRAINING ORCHESTRATOR STARTED at {datetime.datetime.now()} =====\n\n")

# ==========================================
# 2. DATABASE INTEGRATION
# ==========================================
def update_db_status(supabase_url: str, supabase_key: str, client_name: str, status: str):
    if not supabase_url or not supabase_key:
        return
    try:    
        supabase: Client = create_client(supabase_url, supabase_key)
        supabase.from_("clients").update({"status": status}).eq("client_name", client_name).execute()
        log_message("SYSTEM", f"Updated DB status to '{status}' for {client_name}")
    except Exception as e:
        log_message("SYSTEM", f"Failed to update DB status: {e}", level="WARNING")

# ==========================================
# 3. TRAINING CORE
# ==========================================
def modify_last_layer(model, num_classes):
    if num_classes == 1000: return model
    if hasattr(model, 'fc') and isinstance(model.fc, nn.Linear):
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Linear):
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Sequential):
        updated = False
        for i in range(len(model.classifier) - 1, -1, -1):
            if isinstance(model.classifier[i], nn.Linear):
                model.classifier[i] = nn.Linear(model.classifier[i].in_features, num_classes)
                updated = True
                break
        if not updated:
             model.classifier.add_module("final_fc", nn.Linear(1280, num_classes))
    return model

def run_training_background(request_dict: dict):
    client_name = request_dict['client_name']
    dataset_path = request_dict['dataset_path']
    model_path = request_dict['model_path']
    epochs = request_dict['epochs']
    batch_size = request_dict['batch_size']
    lr = request_dict['learning_rate']
    supabase_url = request_dict.get('supabase_url')
    supabase_key = request_dict.get('supabase_key')
    
    log_message(client_name, f"🏋️ Starting training session (Epochs: {epochs}, BS: {batch_size}, LR: {lr})")
    training_logs[client_name]["status"] = "running"
    
    # Update status in DB
    update_db_status(supabase_url, supabase_key, client_name, "Training")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log_message(client_name, f"⚙️ Device: {device}")
    
    try:
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Checkpoint not found: {model_path}")
            
        checkpoint = torch.load(model_path, map_location=device)
        model_name = checkpoint.get('model_type', 'resnet18')
        num_classes = checkpoint.get('num_classes', 2)
        
        log_message(client_name, f"🏗️ Building {model_name}...")
        model = get_model(model_name, weights=None)
        model = modify_last_layer(model, num_classes)
        model.load_state_dict(checkpoint['state_dict'])
        model = model.to(device)
        
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        
        dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)
        log_message(client_name, f"📦 Data Loaded: {len(dataset)} images")
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        model.train()
        for epoch in range(epochs):
            running_loss = 0.0
            for i, (images, labels) in enumerate(dataloader):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                
            avg_loss = running_loss / len(dataloader)
            log_message(client_name, f"📈 Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")
            
        log_message(client_name, "💾 Saving updated model...")
        checkpoint['state_dict'] = model.state_dict()
        torch.save(checkpoint, model_path)
        
        training_logs[client_name]["status"] = "completed"
        log_message(client_name, "✅ Training Successfully Completed")
        update_db_status(supabase_url, supabase_key, client_name, "Active")
        
    except Exception as e:
        error_log = traceback.format_exc()
        log_message(client_name, f"❌ FAILED: {str(e)}", level="ERROR")
        log_message(client_name, error_log, level="DEBUG")
        training_logs[client_name]["status"] = "failed"
        update_db_status(supabase_url, supabase_key, client_name, "Inactive")

# ==========================================
# 4. API ENDPOINTS & APP
# ==========================================
class ClientEnrollment(BaseModel):
    client_name: str
    model_type: str
    num_classes: int = 2

class TrainRequest(BaseModel):
    client_name: str
    dataset_path: str
    model_path: str
    epochs: int = 10
    batch_size: int = 16
    learning_rate: float = 0.001
    simulation_id: Optional[int] = None
    supabase_url: Optional[str] = None
    supabase_key: Optional[str] = None

app = FastAPI(title="FLEX-Med Local Orchestrator")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def health():
    return {"status": "ready", "service": "local-orchestrator", "timestamp": datetime.datetime.now()}

@app.post("/enroll_client")
async def enroll(client: ClientEnrollment):
    try:
        name = client.client_name
        m_type = client.model_type.lower().strip()
        log_message("SYSTEM", f"Enrollment request for {name} ({m_type})")
        
        model = get_model(m_type, weights='DEFAULT')
        model = modify_last_layer(model, client.num_classes)
        
        dest_path = MODELS_DIR / f"{name}.pt"
        torch.save({
            'model_type': m_type,
            'num_classes': client.num_classes,
            'state_dict': model.state_dict()
        }, dest_path)
        
        return {"status": "success", "path": str(dest_path)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/start_local_train")
async def start_train(request: TrainRequest, background_tasks: BackgroundTasks):
    if not os.path.exists(request.dataset_path):
        raise HTTPException(status_code=404, detail=f"Dataset not found: {request.dataset_path}")
    
    # Clear individual logs
    with open(f"{request.client_name}.log", "w") as f:
        f.write(f"--- Log for {request.client_name} started at {datetime.datetime.now()} ---\n")
        
    background_tasks.add_task(run_training_background, request.dict())
    return {"status": "started", "client": request.client_name}

@app.get("/train_status/{name}")
def status(name: str):
    return training_logs.get(name, {"status": "unknown"})

@app.get("/logs")
def get_logs(lines: int = 50):
    if not os.path.exists(CENTRAL_LOG_FILE): return {"logs": []}
    with open(CENTRAL_LOG_FILE, "r") as f:
        return {"logs": f.readlines()[-lines:]}


## **4. Server & Tunnel Management**

In [ ]:
NGROK_TOKEN = "38BI51ntfzceNwdU1BTYDLnAZib_3S3SmcwQJFykk89BGBmEJ" # Replaced with provided token
ngrok.set_auth_token(NGROK_TOKEN)

try:
    # Start Tunnel
    public_url = ngrok.connect(8000).public_url
    print(f"\n🚀 ORCHESTRATOR ONLINE")
    print(f"🌐 URL: {public_url}")
    print(f"🎯 Backend config: Set LOCAL_TRAIN_ORCHESTRATOR_URL={public_url}\n")
    
    # Start FastAPI
    uvicorn.run(app, host="0.0.0.0", port=8000)
except Exception as e:
    print(f"❌ Error: {e}")

## **5. Monitoring & Utilities**

In [ ]:
# View last 50 lines of central log
!tail -n 50 local_training.log

In [ ]:
# Run this cell to kill the server and free port 8000
!fuser -k 8000/tcp
ngrok.kill()